# 第05章 寻优的旅程：计算实验

![图5-1：机器人下山分镜](图片/A2-G19-章节导入.svg)

*图5-1：概念示意，非实测迭代轨迹。*

| 第一格：只看方向 | 第二格：跨过山谷 | 第三格：拿出局部地图 |
| --- | --- | --- |
| 机器人：“箭头朝下，我迈大步就行！” | 同伴：“你一步跨到了更高的山坡！” | 机器人：“先看曲率，试一步，再决定接不接受。” |

*概念分镜；下列图表才是实测迭代。*

机器人沿下坡方向迈大步，却越过山谷落在更高处。方向只说明局部变化，步长决定有限移动的结果。先预测：$\phi(x)=x^2$、$x_0=1$、$\alpha=1.2$ 时，负梯度迭代为何发散？下面分别考察求根、下降、曲率和拟合。

四节与[正文](A2-G19-正文.md)对应；完成后可做[习题](A2-G19-习题.md)，并对照[参考解答](A2-G19-习题参考解答.md)。代码仅使用 NumPy 和 Matplotlib；从空内核顺序执行，无运行时安装。结果写入相对目录，随机种子明确固定。实测结果仅针对列出的数据，不证明一般收敛定理。主要理论参考为文献[1]高立《数值最优化方法》；书目见正文。


In [1]:
import os, json, time, platform, importlib.metadata
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path('A2-G19-数据')/'.matplotlib'))
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_candidates=font_manager.findSystemFonts()
chinese_font=None
for font_hint in ['PingFang','STHeiti Medium','Songti']:
    for candidate in font_candidates:
        if font_hint in candidate:
            try:
                font_manager.fontManager.addfont(candidate)
                plt.rcParams['font.family']=font_manager.FontProperties(fname=candidate).get_name()
                chinese_font=plt.rcParams['font.family'][0]
                break
            except RuntimeError:
                continue
    if chinese_font:break
print('绘图字体:',chinese_font or 'Matplotlib默认字体')
plt.rcParams.update({'font.size':12,'axes.unicode_minus':False,'figure.figsize':(8,5),'lines.linewidth':1.8})
from IPython.display import display
OUT=Path('A2-G19-数据'); IMG=Path('图片')
OUT.mkdir(exist_ok=True); IMG.mkdir(exist_ok=True)
started=time.perf_counter()
results={'versions':{p:importlib.metadata.version(p) for p in ['numpy','matplotlib','nbformat','nbclient']},'python':platform.python_version()}
print(results['versions'], 'Python', results['python'])
def figure(name):
    plt.tight_layout(); plt.savefig(IMG/name,dpi=200); display(plt.gcf()); plt.close()
def show(rows):
    print(json.dumps(rows,ensure_ascii=False,indent=2))


绘图字体: PingFang HK
{'numpy': '2.5.4', 'matplotlib': '3.11.2', 'nbformat': '5.11.1', 'nbclient': '0.11.0'} Python 3.12.10


## 5.1 非线性方程：从夹逼到迭代

求 $f(x)=x^2-2=0$，数学解析根为 $\xi=\sqrt2$；绘图量是与双精度 `sqrt(2)` 基准的差，不把它直接当作真实根误差。二分从 $[1,2]$ 开始，做 $n$ 次区间缩小后中点误差界是 $2^{-n-1}$。压缩映射 $T(x)=(x+2)/(x+1)$ 把 $[1,2]$ 映入自身，并有 $|T'|\le1/4$。Newton 从 1 出发。逐步保存与双精度sqrt2基准的差、残差、步长；另外用标准库Decimal的60位sqrt2核验实际根误差。与双精度基准的差为零只表示两个浮点值相同，不据此估计无限精度收敛阶。

反例 $f(x)=x^3-2x+2$ 从 0 出发精确形成 $0\to1\to0$，小迭代次数本身不构成成功。

In [2]:
from decimal import Decimal, localcontext
with localcontext() as ctx:
    ctx.prec=60
    decimal_root=Decimal(2).sqrt()
    def decimal_error(x):
        with localcontext() as inner:
            inner.prec=60
            return float(abs(Decimal.from_float(float(x))-decimal_root))
xi=np.sqrt(2.)
f=lambda x:x*x-2
hist={}
a,b=1.,2.; prev=None; rows=[]
for n in range(31):
    x=(a+b)/2
    rows.append({'k':n,'x':x,'error':abs(x-xi),'residual':abs(f(x)),'step':None if prev is None else abs(x-prev),'bound':2.**(-n-1)})
    assert abs(x-xi)<=2.**(-n-1)+3*np.finfo(float).eps
    if f(x)>0:b=x
    else:a=x
    prev=x
hist['bisection']=rows
for name,update in [('contraction',lambda x:(x+2)/(x+1)),('newton',lambda x:.5*(x+2/x))]:
    x=1.; rows=[]; prev=None
    for k in range(31 if name=='contraction' else 8):
        rows.append({'k':k,'x':x,'error':abs(x-xi),'residual':abs(f(x)),'step':None if prev is None else abs(x-prev)})
        prev,x=x,update(x)
    hist[name]=rows
cycle=[];x=0.
for k in range(8):
    fx=x**3-2*x+2; step=-fx/(3*x*x-2)
    cycle.append({'k':k,'x':x,'residual':abs(fx),'step':abs(step),'error_to_real_root':abs(x-(-1.7692923542386314))})
    x+=step
assert [v['x'] for v in cycle]==[0.,1.]*4
assert hist['newton'][-1]['residual']<1e-14
for rows in hist.values():
    for row in rows:row['decimal_root_error']=decimal_error(row['x'])
for row in hist['bisection']:assert row['decimal_root_error']<=row['bound']+1e-16
results['5.1']={'error_label':'与双精度sqrt2基准的差','decimal_precision':60,'decimal_sqrt2':str(decimal_root),'history':hist,'cycle':cycle,'stop':'规定次数；Newton到达浮点舍入平台'}
show({name:rows[-1] for name,rows in hist.items()}); show(cycle)
root_styles={'bisection':('二分法','-', 'o'),'contraction':('压缩迭代','--','s'),'newton':('Newton法','-.','^')}
for name,rows in hist.items():
    label,style,marker=root_styles[name]
    plt.semilogy([r['k'] for r in rows],np.maximum([r['error'] for r in rows],1e-17),color='black',linestyle=style,marker=marker,markevery=max(1,len(rows)//7),markersize=5,label=label)
plt.semilogy(np.arange(31),2.**(-np.arange(31)-1),color='.55',linestyle=':',label='二分理论界 2^(-n-1)')
plt.xlabel('区间缩小/更新次数');plt.ylabel('与双精度sqrt2基准的差');plt.legend();figure('A2-G19-51-求根误差.png')
show({'Newton最终Decimal根误差':hist['newton'][-1]['decimal_root_error'],'压缩最终Decimal根误差':hist['contraction'][-1]['decimal_root_error']})


{
  "bisection": {
    "k": 30,
    "x": 1.4142135619185865,
    "error": 4.545086529361697e-10,
    "residual": 1.285544337648048e-09,
    "step": 4.656612873077393e-10,
    "bound": 4.656612873077393e-10,
    "decimal_root_error": 4.5450855626323656e-10
  },
  "contraction": {
    "k": 30,
    "x": 1.4142135623730951,
    "error": 0.0,
    "residual": 4.440892098500626e-16,
    "step": 0.0,
    "decimal_root_error": 9.667293313452913e-17
  },
  "newton": {
    "k": 7,
    "x": 1.414213562373095,
    "error": 2.220446049250313e-16,
    "residual": 4.440892098500626e-16,
    "step": 0.0,
    "decimal_root_error": 1.2537167179050217e-16
  }
}
[
  {
    "k": 0,
    "x": 0.0,
    "residual": 2.0,
    "step": 1.0,
    "error_to_real_root": 1.7692923542386314
  },
  {
    "k": 1,
    "x": 1.0,
    "residual": 1.0,
    "step": 1.0,
    "error_to_real_root": 2.7692923542386314
  },
  {
    "k": 2,
    "x": 0.0,
    "residual": 2.0,
    "step": 1.0,
    "error_to_real_root": 1.7692923542386314

<Figure size 800x500 with 1 Axes>

{
  "Newton最终Decimal根误差": 1.2537167179050217e-16,
  "压缩最终Decimal根误差": 9.667293313452913e-17
}


![图5-2：求根迭代与双精度sqrt2基准的差](图片/A2-G19-51-求根误差.png)

*实测值；差为零仅在绘图时截到1e−17。灰色虚线为二分真实根误差的理论界，界单调下降，实际中点差可起伏。Decimal高精度对照另存于结果表。*

## 5.2 下降方向：往哪里走，走多远

$\phi(x)=\frac12x^TQx$，$Q=\operatorname{diag}(1,20)$，$x_0=(2,2)^T$，解析最优点为零。固定步长必须 $0<\alpha<2/20$；比较 0.05、边界 0.1 与 0.12，并保留失败。Armijo 用 $c=10^{-4}$、缩小因子 1/2，从 $\alpha=1$ 开始。记录每步目标、梯度范数、步长及函数/梯度求值数；停止判断用梯度而非仅步长。


In [3]:
Q=np.diag([1.,20.])
def quad(x):return .5*x@Q@x,Q@x

def descent(mode,steps=500):
    x=np.array([2.,2.]); rows=[];nf=ng=0; reason='达到迭代上限'
    for k in range(steps):
        val,g=quad(x);nf+=1;ng+=1
        if np.linalg.norm(g)<1e-8:reason='梯度范数<1e-8';break
        alpha=float(mode) if mode!='armijo' else 1.
        if mode=='armijo':
            for trial in range(60):
                z=x-alpha*g;candidate=.5*z@Q@z;nf+=1
                if candidate<=val-1e-4*alpha*(g@g):break
                alpha*=.5
            else:reason='回溯失败';break
        rows.append({'k':k,'x':x.tolist(),'phi':float(val),'gradient':float(np.linalg.norm(g)),'alpha':alpha,'step':float(alpha*np.linalg.norm(g)),'nf':nf,'ng':ng})
        x-=alpha*g
        if np.linalg.norm(x)>1e8:reason='发散阈值||x||>1e8';break
    val,g=quad(x);nf+=1;ng+=1
    return {'history':rows,'x':x.tolist(),'phi':float(val),'error':float(np.linalg.norm(x)),'gradient':float(np.linalg.norm(g)),'nf':nf,'ng':ng,'stop':reason}
qres={str(a):descent(a) for a in [.05,.1,.12,'armijo']}
assert qres['0.05']['error']<1e-8 and qres['armijo']['error']<1e-8
assert qres['0.12']['phi']>quad(np.array([2.,2.]))[0]
results['5.2']=qres
show({key:{k:v for k,v in r.items() if k!='history'} for key,r in qres.items()})
qstyles=[('-', 'o'),('--','s'),('-.','^'),(':','D')]
for (key,r),(style,marker) in zip(qres.items(),qstyles):
    label='Armijo回溯' if key=='armijo' else '固定步长 '+key
    plt.semilogy([h['k'] for h in r['history']],[h['phi'] for h in r['history']],color='black',linestyle=style,marker=marker,markevery=max(1,len(r['history'])//8),markersize=4,label=label)
plt.xlabel('更新次数');plt.ylabel('目标函数值');plt.legend();figure('A2-G19-52-下降比较.png')


{
  "0.05": {
    "x": [
      9.8160049427636e-09,
      0.0
    ],
    "phi": 4.817697651817971e-17,
    "error": 9.8160049427636e-09,
    "gradient": 9.8160049427636e-09,
    "nf": 375,
    "ng": 375,
    "stop": "梯度范数<1e-8"
  },
  "0.1": {
    "x": [
      2.6441416389616092e-23,
      2.0
    ],
    "phi": 40.0,
    "error": 2.0,
    "gradient": 40.0,
    "nf": 501,
    "ng": 501,
    "stop": "达到迭代上限"
  },
  "0.12": {
    "x": [
      0.0022835557543318145,
      -111126052.32382658
    ],
    "phi": 1.2348999505077843e+17,
    "error": 111126052.32382658,
    "gradient": 2222521046.4765315,
    "nf": 54,
    "ng": 54,
    "stop": "发散阈值||x||>1e8"
  },
  "armijo": {
    "x": [
      7.731858790301267e-09,
      -2.660631108352562e-10
    ],
    "phi": 3.0598715966052824e-17,
    "error": 7.736435221218252e-09,
    "gradient": 9.386025353262823e-09,
    "nf": 783,
    "ng": 151,
    "stop": "梯度范数<1e-8"
  }
}


<Figure size 800x500 with 1 Axes>

![图5-3：固定步长与Armijo的目标变化](图片/A2-G19-52-下降比较.png)

*实测值；α=.1在边界振荡，α=.12发散，正下降方向仍需合适步长。*

## 5.3 Newton与拟Newton：把曲率用起来

Rosenbrock 函数 $\phi(x)=(1-x_1)^2+100(x_2-x_1^2)^2$ 的解析最优点为 $(1,1)$。Newton 遇到不正定 Hessian 时以 $H+\mu I$ 替换，使最小特征值至少 $10^{-3}$。BFGS 保存 Hessian 近似 $B$，只有 $s^Ty>10^{-10}\|s\|\|y\|$ 时更新，否则跳过；两者都用 Armijo 回溯。本实验**未实现 Wolfe 条件**，不能用 Armijo 宣称曲率条件自动满足。

先验证不定 Hessian 的 Newton 方向可能上升，再比较两种方法的实际成本。

In [4]:
Hbad=np.diag([-1.,2.]);gbad=np.array([1.,0.]);pbad=np.linalg.solve(Hbad,-gbad)
assert gbad@pbad>0
print('不定Hessian反例 g^T p =',gbad@pbad)
xq=np.array([2.,2.]);pq=np.linalg.solve(Q,-Q@xq);xq1=xq+pq
assert np.linalg.norm(xq1)==0
print('正定二次型 Newton 一步误差:',np.linalg.norm(xq1))
def rose(x,need_hessian=True):
    a,b=x;v=(1-a)**2+100*(b-a*a)**2
    g=np.array([2*(a-1)-400*a*(b-a*a),200*(b-a*a)])
    H=np.array([[2-400*b+1200*a*a,-400*a],[-400*a,200.]]) if need_hessian else None
    return v,g,H

def optimize(method):
    x=np.array([-1.2,1.]);B=np.eye(2);rows=[];nf=ng=nh=0;skip=0;reason='达到迭代上限'
    for k in range(500):
        v,g,H=rose(x,method=='newton');nf+=1;ng+=1;nh+=int(method=='newton')
        if np.linalg.norm(g)<1e-8:reason='梯度范数<1e-8';break
        mu=max(0.,1e-3-np.linalg.eigvalsh(H)[0]) if method=='newton' else 0.
        p=np.linalg.solve(H+mu*np.eye(2) if method=='newton' else B,-g)
        assert g@p<0
        alpha=1.
        for trial in range(60):
            z=x+alpha*p;vn=(1-z[0])**2+100*(z[1]-z[0]**2)**2;nf+=1
            if vn<=v+1e-4*alpha*(g@p):break
            alpha*=.5
        else:reason='回溯失败';break
        xn=x+alpha*p;a,b=xn;gn=np.array([2*(a-1)-400*a*(b-a*a),200*(b-a*a)]);ng+=1;s=xn-x;y=gn-g
        accepted=True
        if method=='bfgs':
            accepted=s@y>1e-10*np.linalg.norm(s)*np.linalg.norm(y)
            if accepted:
                Bs=B@s;B=B-np.outer(Bs,Bs)/(s@Bs)+np.outer(y,y)/(s@y);B=(B+B.T)/2
                assert np.linalg.eigvalsh(B)[0]>0
            else:skip+=1
        rows.append({'k':k,'phi':float(v),'gradient':float(np.linalg.norm(g)),'alpha':alpha,'step':float(np.linalg.norm(s)),'mu':float(mu),'curvature':float(s@y),'update':bool(accepted),'nf':nf,'ng':ng,'nh':nh})
        x=xn
    v,g,H=rose(x,False);nf+=1;ng+=1
    return {'history':rows,'x':x.tolist(),'phi':float(v),'error':float(np.linalg.norm(x-1)),'gradient':float(np.linalg.norm(g)),'nf':nf,'ng':ng,'nh':nh,'skipped_updates':skip,'stop':reason}
opt={m:optimize(m) for m in ['newton','bfgs']}
for r in opt.values():assert r['error']<1e-6 and r['phi']<1e-12
results['5.3']={'methods':opt,'indefinite_direction_dot':float(gbad@pbad),'quadratic_newton_one_step':{'x':xq1.tolist(),'error':float(np.linalg.norm(xq1))}}
show({m:{k:v for k,v in r.items() if k!='history'} for m,r in opt.items()})
for m,r in opt.items():
    plt.semilogy([h['k'] for h in r['history']],np.maximum([h['phi'] for h in r['history']],1e-20),color='black',linestyle='-' if m=='newton' else '--',marker='o' if m=='newton' else 's',markevery=4,markersize=4,label='修正Newton' if m=='newton' else 'BFGS')
plt.xlabel('更新次数');plt.ylabel('目标函数值');plt.legend();figure('A2-G19-53-曲率比较.png')


不定Hessian反例 g^T p = 1.0
正定二次型 Newton 一步误差: 0.0
{
  "newton": {
    "x": [
      0.9999999999400667,
      0.9999999998789006
    ],
    "phi": 3.743975643139474e-21,
    "error": 1.35118656593187e-10,
    "gradient": 4.473328123560842e-10,
    "nf": 51,
    "ng": 44,
    "nh": 22,
    "skipped_updates": 0,
    "stop": "梯度范数<1e-8"
  },
  "bfgs": {
    "x": [
      0.9999999999994507,
      0.9999999999989695
    ],
    "phi": 7.664560722213772e-25,
    "error": 1.167784846756675e-12,
    "gradient": 3.147204139240075e-11,
    "nf": 91,
    "ng": 72,
    "nh": 0,
    "skipped_updates": 0,
    "stop": "梯度范数<1e-8"
  }
}


<Figure size 800x500 with 1 Axes>

![图5-4：Rosenbrock函数上的Newton与BFGS](图片/A2-G19-53-曲率比较.png)

*实测目标轨迹；两方法均用Armijo，本图不实施Wolfe。*

## 5.4 非线性最小二乘：从残差到参数

模型 $y_i=a\exp(-bt_i)+\epsilon_i$，$r_i=a\exp(-bt_i)-y_i$，$\Phi=\frac12\|r\|^2$。真实参数 $(2,0.7)$；30 个等距采样点，噪声标准差 0.03。GN 用 `lstsq(J,-r)` 避免形成正规方程；LM 解 $(J^TJ+\lambda I)p=-J^Tr$，预测下降为 $-g^Tp-\frac12\|Jp\|^2$，实际下降为 $\Phi(\theta)-\Phi(\theta+p)$。比值 $\rho$ 大于 $10^{-4}$ 才接受；$\rho<1/4$ 时阻尼乘 4，$\rho>3/4$ 时除 2，其余不变。拒绝后保持参数、增大阻尼并重试。

独立基准对每个 $b$ 显式消去 $a=(v^Ty)/(v^Tv)$，在 $[-4,4]$ 网格定位后做黄金分割局部搜索。它验证本数据范围的拟合结果，不声称求出了任意非线性模型的全局最优。


In [5]:
truth=np.array([2.,.7]);t=np.linspace(0,2,30)
def residual(theta,t,y):
    with np.errstate(over='ignore',invalid='ignore'):
        e=np.exp(-theta[1]*t);r=theta[0]*e-y;J=np.column_stack([e,-theta[0]*t*e])
    return r,J

def fit(method,init,t,y,maxit=250):
    theta=np.array(init,dtype=float);lam=1e-3;rows=[];nf=nj=0;reason='达到迭代上限'
    for k in range(maxit):
        r,J=residual(theta,t,y);nf+=1;nj+=1;v=.5*(r@r);g=J.T@r
        if np.linalg.norm(g)<1e-9:reason='梯度范数<1e-9';break
        p=np.linalg.lstsq(J,-r,rcond=None)[0] if method=='gn' else np.linalg.solve(J.T@J+lam*np.eye(2),-g)
        with np.errstate(over='ignore',invalid='ignore'):
            tn=theta+p;rn=tn[0]*np.exp(-tn[1]*t)-y
        nf+=1;vn=.5*(rn@rn)
        pred=-g@p-.5*np.linalg.norm(J@p)**2;actual=v-vn
        rho=actual/pred if pred>0 and np.isfinite(actual) else -np.inf
        accept=bool(np.isfinite(vn) and (method=='gn' or rho>1e-4))
        rows.append({'k':k,'theta':theta.tolist(),'phi':float(v),'gradient':float(np.linalg.norm(g)),'step':float(np.linalg.norm(p)),'lambda':float(lam),'predicted':float(pred),'actual':float(actual) if np.isfinite(actual) else None,'rho':float(rho) if np.isfinite(rho) else None,'accepted':accept,'nf':nf,'nj':nj})
        if method=='gn' and not np.isfinite(vn):reason='非有限试探值';break
        if accept:theta+=p
        if method=='lm':
            if rho<.25:lam=min(lam*4,1e16)
            elif rho>.75:lam=max(lam/2,1e-15)
        if accept and np.linalg.norm(p)<1e-11*(1+np.linalg.norm(theta)):reason='相对步长<1e-11（另检验梯度）';break
    r,J=residual(theta,t,y);nf+=1;nj+=1
    return {'history':rows,'theta':theta.tolist(),'parameter_error':float(np.linalg.norm(theta-truth)),'residual':float(np.linalg.norm(r)),'phi':float(.5*(r@r)),'gradient':float(np.linalg.norm(J.T@r)),'singular_values':np.linalg.svd(J,compute_uv=False).tolist(),'nf':nf,'nj':nj,'stop':reason,'rejected':sum(not h['accepted'] for h in rows)}

def independent(t,y):
    def obj(b,full=False):
        e=np.exp(-b*t);a=e@y/(e@e);val=.5*np.linalg.norm(a*e-y)**2
        return (np.array([a,b]),val) if full else val
    grid=np.linspace(-4,4,4001);vals=np.array([obj(b) for b in grid]);i=int(np.argmin(vals));assert 0<i<len(grid)-1
    lo,hi=grid[i-1],grid[i+1];ratio=(np.sqrt(5)-1)/2
    c=hi-ratio*(hi-lo);d=lo+ratio*(hi-lo)
    for _ in range(80):
        if obj(c)<obj(d):hi=d;d=c;c=hi-ratio*(hi-lo)
        else:lo=c;c=d;d=lo+ratio*(hi-lo)
    theta,val=obj((lo+hi)/2,True)
    return {'theta':theta.tolist(),'phi':float(val),'search_interval':[-4,4]}
yn=truth[0]*np.exp(-truth[1]*t)
noiseless={m:fit(m,[1.8,.5],t,yn) for m in ['gn','lm']}
noiseless.update({m+'_bad':fit(m,[.1,-2.],t,yn) for m in ['gn','lm']})
for r in noiseless.values():assert r['parameter_error']<1e-7
noisy=yn+np.random.default_rng(2026).normal(0,.03,len(t))
experiments={f'{m}_{s}':fit(m,init,t,noisy) for m in ['gn','lm'] for s,init in [('good',[1.8,.5]),('bad',[.1,-2.])]}
ref=independent(t,noisy)
for key in ['gn_good','gn_bad','lm_good','lm_bad']:assert abs(experiments[key]['phi']-ref['phi'])<1e-9
assert experiments['lm_bad']['rejected']>0
for key in ['lm_good','lm_bad']:
    for row in experiments[key]['history']:
        if row['accepted']:assert row['actual']>0 and row['predicted']>0
multi=[]
for seed in range(2026,2031):
    yy=yn+np.random.default_rng(seed).normal(0,.03,len(t));rr=fit('lm',[.1,-2.],t,yy);rr.pop('history');rr['seed']=seed
    baseline=independent(t,yy);rr['reference']=baseline;assert abs(rr['phi']-baseline['phi'])<1e-8;multi.append(rr)
tn=np.linspace(0,.01,30);ynarrow=truth[0]*np.exp(-truth[1]*tn)+np.random.default_rng(2026).normal(0,.03,len(tn))
narrow=fit('lm',[1.8,.5],tn,ynarrow)
refn=independent(tn,ynarrow);assert abs(narrow['phi']-refn['phi'])<1e-8
results['5.4']={'noiseless':noiseless,'noisy':experiments,'reference':ref,'multiseed':multi,'narrow':narrow,'narrow_reference':refn,'noise_sigma':.03,'seed':2026}
show({key:{k:v for k,v in rr.items() if k!='history'} for key,rr in experiments.items()});show({'reference':ref,'narrow':{k:v for k,v in narrow.items() if k!='history'},'narrow_reference':refn});show(multi)
fig,axes=plt.subplots(1,2,figsize=(8,4.8))
for ax,tt,yy,rr,title in [(axes[0],t,noisy,experiments['lm_bad'],'宽区间 t∈[0,2]'),(axes[1],tn,ynarrow,narrow,'窄区间 t∈[0,0.01]')]:
    theta=rr['theta']
    ax.plot(tt,yy,'o',color='.4',markerfacecolor='white',markersize=5,label='观测值（seed 2026）')
    ax.plot(tt,truth[0]*np.exp(-truth[1]*tt),color='black',linestyle='--',label='真实曲线')
    ax.plot(tt,theta[0]*np.exp(-theta[1]*tt),color='black',linestyle='-',label='LM拟合曲线')
    annotation=f"参数=({theta[0]:.4g}, {theta[1]:.4g})"+chr(10)+f"残差范数={rr['residual']:.4g}"+chr(10)+f"参数误差={rr['parameter_error']:.4g}"
    ax.text(.04,.04,annotation,transform=ax.transAxes,fontsize=11.5,bbox={'facecolor':'white','edgecolor':'.6','alpha':.95})
    ax.set_title(title);ax.set_xlabel('时间 t');ax.set_ylabel('观测/模型值');ax.legend(loc='upper right',fontsize=10.5)
figure('A2-G19-54-参数拟合.png')
for key,style,marker,label in [('gn_good','-','o','GN：好初值'),('gn_bad','--','s','GN：坏初值'),('lm_bad','-.','^','LM：坏初值（含拒绝）')]:
    h=experiments[key]['history'];plt.semilogy([r['k'] for r in h],np.maximum([r['phi'] for r in h],1e-20),color='black',linestyle=style,marker=marker,markersize=4,label=label)
plt.xlabel('试探次数（含拒绝）');plt.ylabel('半残差平方和');plt.legend();figure('A2-G19-54-接受拒绝.png')


{
  "gn_good": {
    "theta": [
      1.9929568709340675,
      0.6954329654827172
    ],
    "parameter_error": 0.008394252278877254,
    "residual": 0.12963782939509058,
    "phi": 0.008402983405135305,
    "gradient": 1.4233212982945068e-10,
    "singular_values": [
      5.46319846316184,
      1.8672405961073673
    ],
    "nf": 10,
    "nj": 6,
    "stop": "梯度范数<1e-9",
    "rejected": 0
  },
  "gn_bad": {
    "theta": [
      1.9929568708705578,
      0.6954329654259837
    ],
    "parameter_error": 0.008394252363031323,
    "residual": 0.12963782939509066,
    "phi": 0.008402983405135317,
    "gradient": 7.149884535815341e-10,
    "singular_values": [
      5.463198463377091,
      1.8672405961436989
    ],
    "nf": 22,
    "nj": 12,
    "stop": "梯度范数<1e-9",
    "rejected": 0
  },
  "lm_good": {
    "theta": [
      1.992956870935181,
      0.6954329654835234
    ],
    "parameter_error": 0.008394252277504287,
    "residual": 0.1296378293950907,
    "phi": 0.00840298340513532,


<Figure size 800x480 with 2 Axes>

<Figure size 800x500 with 1 Axes>

![图5-5：宽窄采样区间的观测、真值与LM拟合](图片/A2-G19-54-参数拟合.png)

*两组实测数据都用种子2026；每个子图列出LM参数、残差范数及参数误差（4位有效数字）。小残差不能替代参数辨识。*

![图5-6：GN与LM的目标和拒绝平台](图片/A2-G19-54-接受拒绝.png)

*实测值；GN坏初值先增大目标，LM拒绝时参数不变，表现为平台。*

### 配套习题的独立小核验

题5.3以 $B=I$、$s=(1,0)^T$、$y=(2,1)^T$ 检查一次BFGS更新，精确基准为 $[[2,1],[1,3/2]]$；另核验负曲率导致不正定。题5.4在真参数处用中心差分核验解析Jacobian，从 $h=10^{-2}$ 到 $10^{-10}$ 观察截断误差下降及舍入影响。噪声与参数无关，差分在同一观测数据上进行。这里矩阵Frobenius范数定义为 $\|A\|_F=(\sum_{i,j}a_{ij}^2)^{1/2}$，差分表记录 $\|J_h-J\|_F/\|J\|_F$；代码 `np.linalg.norm` 对二维数组默认使用该范数。

In [6]:
Bc=np.eye(2);sc=np.array([1.,0.]);yc=np.array([2.,1.])
Bsc=Bc@sc;Bplus=Bc-np.outer(Bsc,Bsc)/(sc@Bsc)+np.outer(yc,yc)/(sc@yc)
expected=np.array([[2.,1.],[1.,1.5]])
np.testing.assert_allclose(Bplus,expected,rtol=0,atol=1e-15)
np.testing.assert_allclose(Bplus@sc,yc,rtol=0,atol=1e-15)
assert np.linalg.eigvalsh(Bplus)[0]>0
sneg=np.array([0.,1.]);yneg=np.array([0.,-1.]);Bsneg=Bc@sneg
Bnegative=Bc-np.outer(Bsneg,Bsneg)/(sneg@Bsneg)+np.outer(yneg,yneg)/(sneg@yneg)
assert np.linalg.eigvalsh(Bnegative)[0]<0
_,analytic=residual(truth,t,noisy);diff=[]
for h in 10.**(-np.arange(2,11)):
    approx=np.empty_like(analytic)
    for j in range(2):
        perturb=np.eye(2)[j]*h
        rp,_=residual(truth+perturb,t,noisy);rm,_=residual(truth-perturb,t,noisy)
        approx[:,j]=(rp-rm)/(2*h)
    error=np.linalg.norm(approx-analytic)/np.linalg.norm(analytic)
    diff.append({'h':float(h),'relative_error':float(error)})
assert min(row['relative_error'] for row in diff)<1e-8
assert diff[-1]['relative_error']>min(row['relative_error'] for row in diff)
results['exercise_checks']={'bfgs_matrix':Bplus.tolist(),'bfgs_eigenvalues':np.linalg.eigvalsh(Bplus).tolist(),'negative_curvature_matrix':Bnegative.tolist(),'jacobian_difference':diff}
show(results['exercise_checks'])


{
  "bfgs_matrix": [
    [
      2.0,
      1.0
    ],
    [
      1.0,
      1.5
    ]
  ],
  "bfgs_eigenvalues": [
    0.7192235935955849,
    2.7807764064044154
  ],
  "negative_curvature_matrix": [
    [
      1.0,
      0.0
    ],
    [
      0.0,
      -1.0
    ]
  ],
  "jacobian_difference": [
    {
      "h": 0.01,
      "relative_error": 2.959785231634601e-05
    },
    {
      "h": 0.001,
      "relative_error": 2.9597421690636323e-07
    },
    {
      "h": 0.0001,
      "relative_error": 2.9596837370961688e-09
    },
    {
      "h": 1e-05,
      "relative_error": 2.7848072289621693e-11
    },
    {
      "h": 1e-06,
      "relative_error": 5.578453961646088e-11
    },
    {
      "h": 1e-07,
      "relative_error": 7.199345441283525e-10
    },
    {
      "h": 1e-08,
      "relative_error": 7.953931721999961e-09
    },
    {
      "h": 1e-09,
      "relative_error": 7.041536219495101e-08
    },
    {
      "h": 1e-10,
      "relative_error": 5.988879754736033e-07
    }
  ]

### 实际运行记录与解释

下一单元生成机器可读完整轨迹。5.1 的残差必须结合导数尺度才能推断根误差，二周期展示初值的重要性。5.2 中过大的固定步长使目标增大；Armijo 的回溯次数也应计入成本。5.3 的函数/梯度/Hessian 求值数只是此实现的计数，不等于不同硬件上的时间排名。5.4 的小残差不意味着小参数误差：含噪声时拟合目标是观测数据，窄采样区间使 Jacobian 的最小奇异值缩小、参数更难辨识。

练习：改变二分初区间；用Decimal基准画出 Newton 相邻根误差之比；把二次型最大的特征值改为 100；统计 BFGS 的曲率筛查；对比多随机种子的参数误差与残差。这些扩展需重新运行，不把当前表格外推成普遍结论。


In [7]:
results['elapsed_seconds']=time.perf_counter()-started
with (OUT/'A2-G19-数值结果.json').open('w',encoding='utf-8') as fh:json.dump(results,fh,ensure_ascii=False,indent=2,allow_nan=False)
print('全部断言通过；结果与图已保存。总用时（秒）',round(results['elapsed_seconds'],3))


全部断言通过；结果与图已保存。总用时（秒） 0.735
